# Signed AHAB Image Generation for U-Boot and Kernel

In this notebook, we'll demonstrate a proof-of-concept on how to:
- Generate signed AHAB U-Boot and kernel images
- Replace the unsigned AHAB U-Boot in the `wic` file
- Replace the unsigned kernel image with a signed version
- Boot up the system with authenticated images


## 1. Prerequisites

- SPSDK is needed with examples extension. `pip install spsdk[examples]` (Please refer to the installation documentation.)
- [Download Embedded Linux for your i.MX board](https://www.nxp.com/design/design-center/software/embedded-software/i-mx-software/embedded-linux-for-i-mx-applications-processors:IMXLINUX)
- This demo was tested with an i.MX95 FRDM and `Linux 6.18.20-2.0.0`


### 1.1 Images preparation
- to create the resulting binary containing AHAB containers, we need to prepare the binaries
- in this section we reproduce the process which is done by the `imx-mkimage` tool as a flash singleboot binary
- Obtain all the necessary binaries and put them into the `inputs` directory

### 1.2 U-Boot

In order to enable secure boot, U-Boot must be built with AHAB support. `CONFIG_AHAB_BOOT=y`
If you want to use nxpele over fastboot, also multiplexing of console output to fastboot must be enabled by setting `CONFIG_CONSOLE_MUX=y`.

As building a (signed) bootloader image for i.MX95 is already explained in the [i.MX95 signed AHAB with U-BOOT](./imx95_signed_ahab_uboot.ipynb) example, we skip the details here.

### 1.3 Requirements

- lpddr4 firmware files
- u-boot binary (u-boot SPL and u-boot) built with AHAB support
- bl31.bin binary (ARM Trusted Firmware)
- ELE firmware binary
- Kernel image (`Image-imx95evk.bin`)
- Device tree blob (`imx95-15x15-frdm.dtb`)
- Bootable disk image `.wic` file for the i.MX95 EVK board
- Four ECC keys and four ML-DSA keys (shared example keys are used from `_data/keys/ecc384` and `_data/keys/mldsa65`)
- i.MX95 FRDM board itself


In [1]:
from spsdk.utils.jupyter_utils import YamlDiffWidget

# This env variable sets colored logger output to STDOUT
%env JUPYTER_SPSDK=1
%alias execute echo %l && %l
%alias_magic ! execute

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


## 2 Building the AHAB Images

We first build the bootloader image, then we build the Kernel image with DTB.


In [2]:
U_BOOT_FLASH_BOOT_CONFIG = "./inputs/configs/bootimg_signed_mx95_serial_downloader.yaml"
U_BOOT_FLASH_BOOT = "./outputs/imx95-ahab-uboot-signed.bin"

In [3]:
%! nxpimage -v bootable-image export --config $U_BOOT_FLASH_BOOT_CONFIG --output $U_BOOT_FLASH_BOOT

nxpimage -v bootable-image export --config ./inputs/configs/bootimg_signed_mx95_serial_downloader.yaml --output ./outputs/imx95-ahab-uboot-signed.bin 
INFO:spsdk.apps.nxpimage_apps.nxpimage_bimg:Created Bootable Image:
Name:      Bootable Image for mimx9596, Revision: latest
Starts:    0x0
Ends:      0x2b3fff
Size:      Size: 2.7 MiB; 2,834,432 B
Alignment: 1 B
Execution Start Address: Not defined
Pattern:zeros
Memory type: MemoryType.SERIAL_DOWNLOADER
Format: bin
INFO:spsdk.apps.nxpimage_apps.nxpimage_bimg:Created Bootable Image memory map:

+==0x0000_0000= Bootable Image for mimx9596, Revision: latest ==+
|                  Size: 2.7 MiB; 2,834,432 B                   |
|           Memory type: MemoryType.SERIAL_DOWNLOADER           |
|                        Pattern: zeros                         |
|+==0x0000_0000= primary_image_container_set ==================+|
||                 Size: 860.0 kiB; 880,640 B                  ||
|+==0x000d_6fff========================================

The Kernel and Device tree blob AHAB image is also very simple, and signing works in a similar fashion.


In [4]:
KERNEL_CONTAINER_CONFIG = "./inputs/configs/mx95_signed_kernel_dtb.yaml"
KERNEL_CONTAINER = "./outputs/os_cntr_signed.bin"
YamlDiffWidget("./inputs/diffs/mx95_signed_kernel_dtb.diffc").html

nxpimage ahab get-template -f mimx9596 -o workspace/ahab_template.yaml --force 
Creating workspace/ahab_template.yaml template file.


In [5]:
%! nxpimage -v ahab export -c $KERNEL_CONTAINER_CONFIG

nxpimage -v ahab export -c ./inputs/configs/mx95_signed_kernel_dtb.yaml 
INFO:spsdk.crypto.keys:Legacy ML-DSA private key detected. SPSDK loaded this expanded-secret format via spsdk_pqc compatibility mode.
INFO:spsdk.crypto.keys:Legacy ML-DSA private key detected. SPSDK loaded this expanded-secret format via spsdk_pqc compatibility mode.
INFO:spsdk.image.ahab.ahab_iae:Image (Linux Kernel Image) size 35678720 bytes adjusted to 35679232 bytes
INFO:spsdk.image.ahab.ahab_iae:Image (Device Tree Blob) size 96394 bytes adjusted to 97280 bytes
INFO:spsdk.image.ahab.ahab_image:AHAB Image Offset has been updated in serial downloader mode.
INFO:spsdk.image.ahab.ahab_image:AHAB Image Offset Updates:
INFO:spsdk.image.ahab.ahab_image:+-----------+-------+------------+------------+------------+-----------+---------+
INFO:spsdk.image.ahab.ahab_image:| Container | Image | Old Offset | New Offset | Image Size | Gap After |  Action |
INFO:spsdk.image.ahab.ahab_image:+-----------+-------+------------+---

## 3 Image Download

First we will download the WIC image onto the board, which by default comes with an unsigned bootloader and Linux kernel (amongst other things like the root filesystem, software, etc.)

It is possible to replace U-Boot with a signed one in the `.wic` file, but the issue with that is that once we do that, the system won't boot, because the signed bootloader will also expect a signed kernel image with a device tree blob.

For completeness the script below would replace the old U-Boot with the new signed one:

```
WIC = "./dist/imx-image-full-imx95evk.wic"
nxpimage bootable-image wic update-uboot -b $WIC -u $U_BOOT_FLASH_BOOT
```

To download the WIC image onto the board we can use the SPSDK utility `nxpuuu`. As the WIC image often has a size of several gigabytes, this process can take several minutes.

We set our i.MX95 board into serial download mode, then use the following command:


In [6]:
WIC = "./dist/imx-image-full-imx95evk.wic"
VERBOSITY = "-v"
%! nxpuuu write $VERBOSITY -f mimx9596 -b emmc_all $U_BOOT_FLASH_BOOT $WIC

nxpuuu write -v -f mimx9596 -b emmc_all ./outputs/imx95-ahab-uboot-signed.bin ./dist/imx-image-full-imx95evk.wic 
Success


## 4 Bootloader replacement
Now that we have uploaded the `.wic` image onto the board, we can proceed with replacing the bootloader with the signed one, as we did not modify the `.wic` image.

Since we just uploaded the `.wic` image onto the board, we should still be in serial download mode.

The following command writes the new bootloader onto the emmc:


In [7]:
%! nxpuuu write -f mimx9596 -b emmc $U_BOOT_FLASH_BOOT

nxpuuu write -f mimx9596 -b emmc ./outputs/imx95-ahab-uboot-signed.bin 
Success


## 5 Kernel replacement

Now that we uploaded the signed bootloader, namely U-Boot which now expects a signed kernel image named `os_cntr_signed.bin` in the FAT partition, we need to upload it.


In [8]:
import os

KERNEL_SIZE = hex(os.path.getsize(KERNEL_CONTAINER))
FASTBOOT_BUFFER_ADDRESS = hex(
    0xA8000000
)  # This is the address which shows when you `echo ${cntr_addr}`
MMCDEV = 0
MMCPART = 1

In [9]:
# We turn on the u-boot console multiplexing into terminal so that we are able to see the output
%! nxpuuu run "FB: ucmd setenv stdout serial,fastboot"
# We set the fastboot buffer to a known address we have access to, in this case 0xA8000000
%! nxpuuu run "FB: ucmd setenv fastboot_buffer $FASTBOOT_BUFFER_ADDRESS"
# Download the kernel
%! nxpuuu run "FB: download -f $KERNEL_CONTAINER"
# Write the kernel to the FAT partition, these parameters may vary if we use an SD card etc.
%! nxpuuu run "FB: ucmd fatwrite mmc $MMCDEV:$MMCPART $FASTBOOT_BUFFER_ADDRESS os_cntr_signed.bin $KERNEL_SIZE"

nxpuuu run "FB: ucmd setenv stdout serial,fastboot" 
Response: Detect USB boot. Will enter fastboot mode!
(NULL udevice *): request 00000000fdee2980 was not queued to ep1in-bulk

Success
nxpuuu run "FB: ucmd setenv fastboot_buffer 0xa8000000" 
Response: Detect USB boot. Will enter fastboot mode!
(NULL udevice *): request 00000000fdee2980 was not queued to ep1in-bulk

Success
nxpuuu run "FB: download -f ./outputs/os_cntr_signed.bin" 
Response: Starting download of 35825664 bytes
(NULL udevice *): request 00000000fdee2980 was not queued to ep1in-bulk
..........................................................................
..........................................................................
..........................................................................
...................................................
downloading of 35825664 bytes finished
(NULL udevice *): request 00000000fdee2980 was not queued to ep1in-bulk

Success
nxpuuu run "FB: ucmd fatwrite mmc 0:1 0xa8000000

We can now delete the old unsigned kernel. Note that this step is optional, but the signed bootloader should not be loading an unsigned kernel by default.


In [10]:
%! nxpuuu -v run "FB: ucmd fatrm mmc 0:1 Image"

nxpuuu -v run "FB: ucmd fatrm mmc 0:1 Image" 
Response: Detect USB boot. Will enter fastboot mode!
(NULL udevice *): request 00000000fdee2980 was not queued to ep1in-bulk

Success


We can list the fat partition and see the various device tree blobs, other binaries, and most importantly our `os_cntr_signed.bin` file.


In [11]:
%! nxpuuu -v run "FB: ucmd fatls mmc 0:1"

nxpuuu -v run "FB: ucmd fatls mmc 0:1" 
Response:     66417   imx95-15x15-ab2.dtb
    99267   imx95-15x15-evk-adv7535-ap1302.dtb
    96570   imx95-15x15-evk.dtb
    77273   imx95-15x15-frdm-8mic-reve.dtb
    98239   imx95-15x15-frdm-ap1302.dtb
    79216   imx95-15x15-frdm-aud-hat.dtb
    76318   imx95-15x15-frdm-boe-wxga-lvds-panel.dtb
    97763   imx95-15x15-frdm-dual-os08a20-isp.dtb
    96530   imx95-15x15-frdm-neutron.dtb
    97151   imx95-15x15-frdm-os08a20-combo-isp.dtb
    97083   imx95-15x15-frdm-os08a20-isp.dtb
    96478   imx95-15x15-frdm-root.dtb
    75076   imx95-15x15-frdm-rpmsg.dtb
    97518   imx95-15x15-frdm-waveshare-7inch-c-panel.dtb
    96394   imx95-15x15-frdm.dtb
   101879   imx95-19x19-evk-adv7535-ap1302.dtb
   100041   imx95-19x19-evk-adv7535.dtb
     2931   imx95-19x19-evk-adv7535.dtbo
   101111   imx95-19x19-evk-ap1302.dtb
     4300   imx95-19x19-evk-ap1302.dtbo
   100327   imx95-19x19-evk-cs42888.dtb
     3429   imx95-19x19-evk-cs42888.dtbo
   101850   imx95-19

## 6 Boot of authenticated container

Now that everything is done, we can change the boot mode to EMMC and see that the authenticated container is booting.

To check if the authenticated container is being booted, we deleted the original unsigned kernel `Image`.

During the boot we should see the following message in the console:

```
Running BSP bootcmd ...
switch to partitions #0, OK
mmc0(part 0) is current device
Failed to load 'boot.scr'
35807506 bytes read in 132 ms (258.7 MiB/s)
Booting from mmc ...
Authenticate OS container at 0x98000000
mu receive msg wait 1s
mu receive msg wait 2s
## Flattened Device Tree blob at 83000000
   Booting using the fdt blob at 0x83000000
Working FDT set to 83000000
   Using Device Tree in place at 0000000083000000, end 000000008300dfbc
Working FDT set to 83000000

Starting kernel ...

[    0.000000] Booting Linux on physical CPU 0x0000000000 [0x412fd050]
```

U-Boot should authenticate and start the kernel automatically, but if it does not, or if you did not download the `.wic` with the root filesystem onto the board, you can run the command `run bootcmd` in the U-Boot console, which should read the `os_cntr_signed.bin` from the FAT partition, authenticate it, and start executing it.
